## C1 - WEEK 2

- Adrià Cantarero Carreras  (1809773@uab.cat)
- Ivan Martin Campoy (1673246@uab.cat)
- Irene Pumares Benaiges (1810704@uab.cat)
- Lavish Ramchandani (1691238@uab.cat)

In [1]:
#needed libraries

import os
import re
import cv2
import pickle
import numpy as np
from functools import partial
from tqdm.auto import tqdm
import time



## TASK 1: Implement 3D / 2D and block and hierarchical histograms

In [2]:
#database and qsd1 and qsd2 paths

bbdd_path = "./BBDD"
qsd2_path = "./qsd2_w1"
qsd1_path = "./qsd1_w1"

COLOR_SPACES = {
    "GRAY": cv2.COLOR_BGR2GRAY,
    "RGB":  cv2.COLOR_BGR2RGB,
    "HSV":  cv2.COLOR_BGR2HSV,
    "LAB":  cv2.COLOR_BGR2LAB,
    "YCRCB": cv2.COLOR_BGR2YCrCb,
}
#----------------------------------------------------------------------------------

CHANNEL_RANGES = {
    "GRAY":  [(0, 256)],
    "RGB":   [(0, 256)] * 3,
    "HSV":   [(0, 180), (0, 256), (0, 256)],
    "LAB":   [(0, 256)] * 3,
    "YCRCB": [(0, 256)] * 3,
}

def read_image(img_path):
    image = cv2.imread(img_path)
    if image is None:
        raise IOError(f"Could not read {img_path}")   #skipping it would shift every index after it and break the ground truth
    return image

bbdd_descriptors = []
qsd2_descriptors = []

#----------------------------------------------------------------------------------
#sort files
def return_number(string):
    return [int(text) if text.isdigit() else text.lower() for text in re.split(r'(\d+)', string)]

#sort bbdd files
bbdd_files = [f for f in os.listdir(bbdd_path) if f.lower().endswith(".jpg")]
bbdd_files.sort(key=return_number)

#sort qsd2 files
qsd2_files = [f for f in os.listdir(qsd2_path) if f.lower().endswith(".jpg")]
qsd2_files.sort(key=return_number)

qsd1_files = [f for f in os.listdir(qsd1_path) if f.lower().endswith(".jpg")]
qsd1_files.sort(key=return_number)

#id of an image = number in its file name (bbdd_00007.jpg -> 7)
def image_id(file_name):
    return int(re.findall(r'\d+', file_name)[-1])

bbdd_ids = np.array([image_id(f) for f in bbdd_files])
qsd2_ids = np.array([image_id(f) for f in qsd2_files])
qsd1_ids = np.array([image_id(f) for f in qsd1_files])
#----------------------------------------------------------------------------------

#load images iteratively bbdd set
bbdd_imgs = []
for file_name in bbdd_files:
    img_path = os.path.join(bbdd_path, file_name)
    image = read_image(img_path)
    bbdd_imgs.append(image)

qsd1_imgs = []
for file_name in qsd1_files:
    img_path = os.path.join(qsd1_path, file_name)
    image = read_image(img_path)
    qsd1_imgs.append(image)

##load images iteratively qsd2 set
qsd2_imgs = []
for file_name in qsd2_files:
    img_path = os.path.join(qsd2_path, file_name)
    image = read_image(img_path)
    qsd2_imgs.append(image)

In [3]:
def compute_nd_histogram(image,color_space = 'HSV', channels = (1,2), bins= 16, mask = None):
    if mask is not None:
        mask = (mask > 0).astype(np.uint8) * 255 #convert boolean mask to 0-255 just in case

    ranges = [] #ranges for the color space channels
    if color_space in COLOR_SPACES:
        converted_image = cv2.cvtColor(image, COLOR_SPACES[color_space])
        if len(converted_image.shape) == 2:
            raise ValueError("Descriptor needs 2 channels at least")

        for c in channels:
            ranges.extend(CHANNEL_RANGES[color_space][c])

    else:
        raise ValueError("Unknown color space: %s" % color_space)

    if isinstance(bins, int):
        hist_size = [bins for i in channels] #ex 16 -> (16,16)
    else:
        assert len(bins) == len(channels), "Number of bins must match number of channels"
        hist_size = bins
  
    hist = cv2.calcHist([converted_image], list(channels), mask, hist_size, ranges)
    hist = hist.flatten()

    #normalize only once
    total = np.sum(hist)
    if total > 0:
        hist = hist / total
    return hist


image = cv2.imread("./BBDD/bbdd_00000.jpg")
d = compute_nd_histogram(image)


In [4]:
def compute_block_histogram(image, grid=(4, 4), color_space='HSV', channels=(1, 2), bins=16, mask=None):
    # 1. block boundaries
    rows, cols = grid
    height, width = image.shape[:2]
    y_cuts  = np.linspace(0, height, rows + 1, dtype=int)
    x_cuts  = np.linspace(0, width, cols + 1, dtype=int)
    block_histograms = []

    # 2. loop over blocks: slice image (and mask), call compute_nd_histogram
    for r in range(rows):
        for c in range(cols):
            y_start, y_end = y_cuts[r], y_cuts[r+1]
            x_start, x_end = x_cuts[c], x_cuts[c+1]
            block_img = image[y_start:y_end, x_start:x_end]
            block_mask = None
            if mask is not None:
                block_mask = mask[y_start:y_end, x_start:x_end]
        
            hist = compute_nd_histogram(block_img, color_space, channels, bins, block_mask)
            block_histograms.append(hist)

    # 3. concatenate (and decide the normalization)
    descriptor = np.concatenate(block_histograms)
    total = np.sum(descriptor)
    if total > 0:
        descriptor = descriptor / total
    return descriptor

image = cv2.imread("./BBDD/bbdd_00000.jpg")
d = compute_block_histogram(image)


In [5]:
#pyramid histogram function
def compute_pyramid_histogram(image, levels=((1, 1), (2, 2), (4, 4)), weights=None,
                              color_space='HSV', channels=(0, 1), bins=16, mask=None):
    # 1. weights: if None, decide a default (one weight per level)
    if weights is None:
        weights = np.ones(len(levels))

    # 2. validate: len(weights) must match len(levels)
    assert len(weights) == len(levels), "len(weights) must match len(levels)"
    level_descriptors = []

    for grid, w in zip(levels, weights):
        # a) descriptor of this level -> compute_block_histogram(...)  (it already sums to 1)
        level_descriptor = compute_block_histogram(image, grid, color_space, channels, bins, mask)
        # b) multiply it by the weight of the level
        level_descriptor = w * level_descriptor
        # c) store it
        level_descriptors.append(level_descriptor)

    # 3. concatenate all levels
    # 4. normalize ONCE so the final vector sums to 1 (check the case total == 0)
    level_descriptors = np.concatenate(level_descriptors)
    total = np.sum(level_descriptors)
    if total > 0:
        level_descriptors = level_descriptors / total
    return level_descriptors

## Task2- Test query system using query set QSD1-W2 development, evaluate retrievalresults (use your best performing descriptor). 
Compare results against the best descriptor/distance in W1 (the dev set is the same)

In [6]:
eps = 1e-10  # small value to avoid division by zero


def euclidean_distance(x, y):
    return np.sqrt(np.sum((x - y) ** 2, axis=-1))

def L1_distance(x, y):
    return np.sum(np.abs(x - y), axis=-1)

def chi_squared_distance(x, y):
    return 0.5 * np.sum(((x - y) ** 2) / (x + y + eps), axis=-1)

def jensen_shannon_distance(x, y):
    m = 0.5 * (x + y)
    kl_x = np.sum(x * np.log((x + eps) / (m + eps)), axis=-1)
    kl_y = np.sum(y * np.log((y + eps) / (m + eps)), axis=-1)
    return np.sqrt(np.clip(0.5 * (kl_x + kl_y), 0, None))

#all the measures below are written as distances (smaller = more similar) because the retrieval step will rank every measure 
#with the same rule (sort ascending and take the first k)

def histogram_intersection_distance(x, y):
    return 1.0 - np.sum(np.minimum(x, y), axis=-1)

def hellinger_distance(x, y):
    return (1.0 / np.sqrt(2.0)) * np.sqrt(np.sum((np.sqrt(np.clip(x, 0, None)) - np.sqrt(np.clip(y, 0, None))) ** 2, axis=-1))

def cosine_distance(x, y):
    return 1.0 - np.sum(x * y, axis=-1) / (np.linalg.norm(x, axis=-1) * np.linalg.norm(y, axis=-1) + eps)

def correlation_distance(x, y):
    xc = x - np.mean(x, axis=-1, keepdims=True)
    yc = y - np.mean(y, axis=-1, keepdims=True)
    return 1.0 - np.sum(xc * yc, axis=-1) / (np.linalg.norm(xc, axis=-1) * np.linalg.norm(yc, axis=-1) + eps)

MEASURES = {
    "euclidean":      euclidean_distance,
    "L1":             L1_distance,
    "chi2":           chi_squared_distance,
    "intersection":   histogram_intersection_distance,
    "hellinger":      hellinger_distance,
    "cosine":         cosine_distance,
    "correlation":    correlation_distance,
    "jensen_shannon": jensen_shannon_distance,
}

# import ml_metrics as metrics #Couldn't install it so I copied the code from the git

import numpy as np

#code from https://github.com/benhamner/Metrics/blob/master/Python/ml_metrics/custom/kdd_average_precision.py
def kdd_apk(actual, predicted, k=10):
    """
    Computes the average precision at k for Track 1 of the 2012 KDD Cup.

    This modified version uses the number of actual clicks as the denominator,
    regardless of k.

    This function computes the average prescision at k between two lists of
    items.

    Parameters
    ----------
    actual : list
             A list of elements that are to be predicted (order doesn't matter)
    predicted : list
                A list of predicted elements (order does matter)
    k : int, optional
        The maximum number of predicted elements

    Returns
    -------
    score : double
            The average precision at k over the input lists

    """
    if len(predicted)>k:
        predicted = predicted[:k]

    score = 0.0
    num_hits = 0.0

    for i,p in enumerate(predicted):
        if p in actual and p not in predicted[:i]: #p not in predicted[:i] is to avoid no extra marks for repeating
            num_hits += 1.0
            score += num_hits / (i+1.0)

    if not actual:
        return 0.0

    return score / len(actual)

def kdd_mapk(actual, predicted, k=10):
    """
    Computes the mean average precision at k for Track 1 of the 2012 KDD Cup.

    This modified version uses the number of actual clicks as the denominator,
    regardless of k.

    This function computes the mean average prescision at k between two lists
    of lists of items.

    Parameters
    ----------
    actual : list
             A list of lists of elements that are to be predicted 
             (order doesn't matter in the lists)
    predicted : list
                A list of lists of predicted elements
                (order matters in the lists)
    k : int, optional
        The maximum number of predicted elements

    Returns
    -------
    score : double
            The mean average precision at k over the input lists

    """
    return np.mean([kdd_apk(a,p,k) for a,p in zip(actual, predicted)])


#apk is like the average precision of hits at k items
#but mapk looks in the entire dataset

import pandas as pd
from IPython.display import display

gt_path = os.path.join(qsd1_path, "gt_corresps.pkl")
with open(gt_path, "rb") as f:
    gt_corresps = pickle.load(f)

num_queries = len(gt_corresps)
assert num_queries == len(qsd1_files), (
    f"GT has {num_queries} entries but QSD1 has {len(qsd1_files)} images."
)
assert len(bbdd_ids) == len(bbdd_files), "BBDD IDs do not match the database files."
actual_ids = [[int(item) for item in query_gt] for query_gt in gt_corresps]


In [7]:


def compute_descriptors(images, descriptor_fn):
    # preallocated matrix (N, D): avoids the list -> array copy that doubles peak memory
    first = np.asarray(descriptor_fn(images[0]), dtype=np.float32)
    descriptors = np.empty((len(images), first.size), dtype=np.float32)
    descriptors[0] = first
    for i in range(1, len(images)):
        descriptors[i] = descriptor_fn(images[i])
    return descriptors

from utils.extra_descriptors import retrieve_topk
def evaluate_descriptors(db_desc, query_desc, measures):
    # db_desc: (287, D) matrix from compute_descriptors on BBDD
    # query_desc: (30, D) matrix from compute_descriptors on QSD1
    # measures: dict {name: distance function}, e.g. MEASURES

    results = []
    for name, measure in measures.items():
        # 1. skip "emd" (it is not meaningful for flattened 2D/3D histograms)
        # 2. top-5 predictions for all queries -> retrieve_topk(...)
        predictions = retrieve_topk(query_desc, db_desc, bbdd_ids, measure, k=5)
        # 3. mAP@1: use only the first prediction of each query -> kdd_mapk(...)
        map1 = kdd_mapk(actual_ids, predictions, k = 1)
        # 4. mAP@5: use the five predictions -> kdd_mapk(...)
        map5 = kdd_mapk(actual_ids, predictions, k = 5)
        # 5. store a dict with distance name, MAP@1 and MAP@5
        results.append({"Distance": name, "MAP@1": map1, "MAP@5": map5})
    return results

In [8]:
import pandas as pd
from functools import partial

# 1. hyperparameters
color_spaces = ["HSV", "LAB", "YCRCB", "RGB"]
PAIRS_2D = {"HSV": (0, 1), "LAB": (1, 2), "YCRCB": (1, 2)}

bins_list = [8, 16, 32]

#configs
block_grids = [(2, 2), (4, 4), (8, 8)]
pyramid_levels = [((1, 1), (2, 2)), ((1, 1), (2, 2), (4, 4))]

grid_experiments = []

def pyramid_weights(scheme, n_levels):
    if scheme == "equal":
        return np.ones(n_levels) #assign equal weight (1.0) to all levels
    if scheme == "fine":
        return 2.0 ** np.arange(n_levels)  #exponential growth (1, 2, 4, etc): gives more weight to fine levels
    raise ValueError(f"Unknown weight scheme: {scheme}")

# 2. grid Search for  block descriptors
def run_grid_search(color_spaces=("HSV", "LAB", "YCRCB", "RGB"), bins_3d=(8, 16), bins_2d=(16, 32),
                    block_grids=((2, 2), (4, 4), (8, 8)),
                    pyramid_levels=(((1, 1), (2, 2)), ((1, 1), (2, 2), (4, 4))),
                    weight_schemes=("equal", "fine"), skip_measures=("intersection",),
                    results_path=os.path.join("results", "Week 2 results", "grid_search_w2_full.csv")):
    # needs: bbdd_imgs, qsd1_imgs, MEASURES, evaluate_descriptors, compute_block_histogram
    # intersection is skipped by default: for L1-normalized histograms it gives the same ranking as L1
    measures = {k: v for k, v in MEASURES.items() if k not in skip_measures}

    # 1. a "family" = (color space, channels, bins); its block descriptors are computed once
    #    and reused by every block grid and every pyramid of that family
    families = []
    for cs in color_spaces:
        for b in bins_3d:
            families.append((cs, (0, 1, 2), b))
        if cs in PAIRS_2D:
            for b in bins_2d:
                families.append((cs, PAIRS_2D[cs], b))
    families.sort(key=lambda f: f[2] ** len(f[1]))  # cheap families first

    # 2. resume: families already saved in the CSV are skipped (a crash does not lose the work)
    rows, done = [], set()
    if os.path.exists(results_path):
        old = pd.read_csv(results_path)
        rows, done = old.to_dict("records"), set(old["Family"])
    os.makedirs(os.path.dirname(results_path) or ".", exist_ok=True)

    grids_needed = sorted(set(block_grids) | {g for levels in pyramid_levels for g in levels})
    pbar = tqdm(families, desc="Grid search")

    for cs, channels, b in pbar:
        family = f"{cs}|{''.join(map(str, channels))}|{b}"
        if family in done:
            continue
        t0, nd = time.time(), f"{len(channels)}D"

        # 3. block descriptors of every grid (BBDD and queries with the SAME function)
        cache = {}
        for grid in grids_needed:
            pbar.set_postfix_str(f"{family} | descriptors {grid[0]}x{grid[1]}")
            fn = partial(compute_block_histogram, grid=grid, color_space=cs, channels=channels, bins=b)
            cache[grid] = (compute_descriptors(bbdd_imgs, fn), compute_descriptors(qsd1_imgs, fn))

        def evaluate(desc_type, config, grid_levels, weights_name, db, q):
            pbar.set_postfix_str(f"{family} | evaluating {config}")
            for res in evaluate_descriptors(db, q, measures):
                rows.append({"Type": desc_type, "Config": config, "Family": family, "Color Space": cs,
                            "Channels": str(channels), "Bins": b, "Grid/Levels": grid_levels,
                            "Weights": weights_name, "Dim": db.shape[1], "Distance": res["Distance"],
                            "MAP@1": res["MAP@1"], "MAP@5": res["MAP@5"]})

        # 4. blocks
        for grid in block_grids:
            evaluate("Block", f"Block {grid[0]}x{grid[1]} | {cs} {nd} | Bins: {b}", str(grid), "-", *cache[grid])

        # 5. pyramids: weighted concatenation of the cached levels (no recomputation of histograms)
        for levels in pyramid_levels:
            levels_str = "->".join(f"{g[0]}x{g[1]}" for g in levels)
            for scheme in weight_schemes:
                w = pyramid_weights(scheme, len(levels))
                parts = [float(wi / w.sum()) for wi in w]  # python floats keep float32
                db = np.hstack([p * cache[g][0] for p, g in zip(parts, levels)])
                q = np.hstack([p * cache[g][1] for p, g in zip(parts, levels)])
                evaluate("Pyramid", f"Pyramid ({levels_str}) [{scheme}] | {cs} {nd} | Bins: {b}",
                        str(levels), scheme, db, q)

        del cache  # free memory before the next family
        pd.DataFrame(rows).to_csv(results_path, index=False)  # checkpoint
        pbar.write(f"{family}: {time.time() - t0:.0f}s")

    df = pd.DataFrame(rows).sort_values(["MAP@1", "MAP@5"], ascending=False).reset_index(drop=True)
    df.to_csv(results_path, index=False)
    return df


df_grid_all = run_grid_search()

print("\n=== TOP 10 GLOBAL CONFIGS FROM THE GRID SEARCH ===")
display(df_grid_all.head(10))

#5. Extract winner and check with week 1 winner
best_grid_winner = df_grid_all.iloc[0]

w1_best_map1 = 0.6333
w1_best_map5 = 0.6872

comparison_table = [
    {
        "Week": "Week 1 (Best Baseline)",
        "Descriptor / Method": "Global 1D CLAHE + LAB",
        "Distance Metric": "L1",
        "mAP@1 QSD1": w1_best_map1,
        "mAP@5 QSD1": w1_best_map5,
        "Gain mAP@1": "-"
    },
    {
        "Week": "Week 2 (Grid Search Winner)",
        "Descriptor / Method": best_grid_winner["Config"],
        "Distance Metric": best_grid_winner["Distance"],
        "mAP@1 QSD1": round(best_grid_winner["MAP@1"], 4),
        "mAP@5 QSD1": round(best_grid_winner["MAP@5"], 4),
        "Gain mAP@1": f"+{round(best_grid_winner['MAP@1'] - w1_best_map1, 4):.4f}"
    }
]

df_comparison_final = pd.DataFrame(comparison_table)

print("\n=== Final comparison: Best W1 VS winner of the grid search of W2 ===")
display(df_comparison_final)

Grid search:   0%|          | 0/14 [00:00<?, ?it/s]

HSV|01|16: 21s
LAB|12|16: 24s
YCRCB|12|16: 20s
HSV|012|8: 34s
LAB|012|8: 38s
YCRCB|012|8: 36s
RGB|012|8: 35s
HSV|01|32: 67s
LAB|12|32: 70s
YCRCB|12|32: 70s
HSV|012|16: 238s
LAB|012|16: 260s
YCRCB|012|16: 236s
RGB|012|16: 270s

=== TOP 10 GLOBAL CONFIGS FROM THE GRID SEARCH ===


,Type,Config,Family,Color Space,Channels,Bins,Grid/Levels,Weights,Dim,Distance,MAP@1,MAP@5
0,Block,Block 8x8 | YCRCB 2D | Bins: 16,YCRCB|12|16,YCRCB,"(1, 2)",16,"(8, 8)",-,16384,chi2,0.833333,0.850000
1,Block,Block 8x8 | YCRCB 2D | Bins: 16,YCRCB|12|16,YCRCB,"(1, 2)",16,"(8, 8)",-,16384,hellinger,0.800000,0.841667
2,Block,Block 8x8 | LAB 2D | Bins: 16,LAB|12|16,LAB,"(1, 2)",16,"(8, 8)",-,16384,jensen_shannon,0.800000,0.834444
3,Block,Block 8x8 | YCRCB 2D | Bins: 16,YCRCB|12|16,YCRCB,"(1, 2)",16,"(8, 8)",-,16384,euclidean,0.800000,0.833333
4,Block,Block 8x8 | YCRCB 2D | Bins: 16,YCRCB|12|16,YCRCB,"(1, 2)",16,"(8, 8)",-,16384,L1,0.800000,0.833333
5,Block,Block 8x8 | YCRCB 2D | Bins: 16,YCRCB|12|16,YCRCB,"(1, 2)",16,"(8, 8)",-,16384,jensen_shannon,0.800000,0.833333
6,Block,Block 8x8 | RGB 3D | Bins: 16,RGB|012|16,RGB,"(0, 1, 2)",16,"(8, 8)",-,262144,chi2,0.800000,0.833333
7,Block,Block 8x8 | LAB 2D | Bins: 16,LAB|12|16,LAB,"(1, 2)",16,"(8, 8)",-,16384,hellinger,0.800000,0.825000
8,Block,Block 8x8 | LAB 2D | Bins: 32,LAB|12|32,LAB,"(1, 2)",32,"(8, 8)",-,65536,hellinger,0.800000,0.816667
9,Block,Block 8x8 | LAB 2D | Bins: 16,LAB|12|16,LAB,"(1, 2)",16,"(8, 8)",-,16384,chi2,0.766667,0.819444



=== Final comparison: Best W1 VS winner of the grid search of W2 ===


,Week,Descriptor / Method,Distance Metric,mAP@1 QSD1,mAP@5 QSD1,Gain mAP@1
0,Week 1 (Best Baseline),Global 1D CLAHE + LAB,L1,0.6333,0.6872,-
1,Week 2 (Grid Search Winner),Block 8x8 | YCRCB 2D | Bins: 16,chi2,0.8333,0.8500,+0.2000


In [9]:

#saving path
output_dir = os.path.join("results", "Week 2 results")
os.makedirs(output_dir, exist_ok=True)

#save grid search results
df_grid_all.to_csv(os.path.join(output_dir, "grid_search_results_w2.csv"), index=False)

#save W1 vs W2 comparison table
df_comparison_final.to_csv(os.path.join(output_dir, "comparison_w1_vs_w2.csv"), index=False)

print("Files saved successfully as CSV in 'results/Week 2 results'!")

Files saved successfully as CSV in 'results/Week 2 results'!


In [10]:
# 1. Define the BEST performing descriptor found during Grid Search (Week 2)
best_w2_descriptor = partial(
    compute_block_histogram,
    grid=(8, 8),
    color_space='YCRCB',
    channels=(1, 2),
    bins=16
)

# 2. Extract feature descriptors for BBDD and QSD1-W2 datasets
print("Extracting descriptors for BBDD using the best W2 model...")
db_desc = compute_descriptors(bbdd_imgs, best_w2_descriptor)

print("Extracting descriptors for QSD1-W2 using the best W2 model...")
query_desc = compute_descriptors(qsd1_imgs, best_w2_descriptor)

# 3. Evaluate retrieval performance across all distance measures
results_best = evaluate_descriptors(db_desc, query_desc, MEASURES)

# 4. Display results sorted by MAP@1
df_results_best = pd.DataFrame(results_best).sort_values(by="MAP@1", ascending=False)
print("=== BEST DESCRIPTOR RESULTS IN W2 (Block 8x8 | YCrCb 2D | Bins 16) ===")
display(df_results_best)

# 5. DIRECT COMPARISON: BEST DESCRIPTOR/DISTANCE IN W1 VS BEST IN W2
w1_best_map1 = 0.6333
w1_best_map5 = 0.6872

best_w2_row = df_results_best.iloc[0]

comparison_table = [
    {
        "Week": "Week 1 (Best Baseline)",
        "Descriptor / Method": "Global 1D CLAHE + LAB",
        "Distance Metric": "L1",
        "mAP@1 QSD1": w1_best_map1,
        "mAP@5 QSD1": w1_best_map5,
        "Gain mAP@1": "-"
    },
    {
        "Week": "Week 2 (Best Proposed)",
        "Descriptor / Method": "Block 8x8 | YCRCB 2D | Bins: 16",
        "Distance Metric": best_w2_row["Distance"],
        "mAP@1 QSD1": round(best_w2_row["MAP@1"], 4),
        "mAP@5 QSD1": round(best_w2_row["MAP@5"], 4),
        "Gain mAP@1": f"+{round(best_w2_row['MAP@1'] - w1_best_map1, 4):.4f}"
    }
]

df_task2_comparison = pd.DataFrame(comparison_table)
print("\n=== TASK 2 COMPARISON: BEST W1 VS BEST W2 ===")
display(df_task2_comparison)

Extracting descriptors for BBDD using the best W2 model...
Extracting descriptors for QSD1-W2 using the best W2 model...
=== BEST DESCRIPTOR RESULTS IN W2 (Block 8x8 | YCrCb 2D | Bins 16) ===


,Distance,MAP@1,MAP@5
2,chi2,0.833333,0.850000
0,euclidean,0.800000,0.833333
1,L1,0.800000,0.833333
3,intersection,0.800000,0.833333
4,hellinger,0.800000,0.841667
7,jensen_shannon,0.800000,0.833333
5,cosine,0.766667,0.791667
6,correlation,0.766667,0.791667



=== TASK 2 COMPARISON: BEST W1 VS BEST W2 ===


,Week,Descriptor / Method,Distance Metric,mAP@1 QSD1,mAP@5 QSD1,Gain mAP@1
0,Week 1 (Best Baseline),Global 1D CLAHE + LAB,L1,0.6333,0.6872,-
1,Week 2 (Best Proposed),Block 8x8 | YCRCB 2D | Bins: 16,chi2,0.8333,0.8500,+0.2000
